# Telegram Export Analysis Notebook

This notebook loads a Telegram `result.json` export, normalizes messages, and produces compact analytics with a short final summary.

## 1) Setup

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse

URL_RE = re.compile(r"https?://\S+", re.IGNORECASE)
HASHTAG_RE = re.compile(r"#\w+", re.UNICODE)
WORD_RE = re.compile(r"[A-Za-zА-Яа-яЁё0-9_]+", re.UNICODE)

def flatten_text(text_field):
    if isinstance(text_field, str):
        return text_field
    if isinstance(text_field, list):
        parts = []
        for item in text_field:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(str(item.get("text", "")))
        return "".join(parts)
    if isinstance(text_field, dict):
        return str(text_field.get("text", ""))
    return ""


## 2) Load Export

In [ ]:
EXPORT_PATH = Path.home() / "Downloads" / "Telega Desktop" / "ChatExport_2026-02-23" / "result.json"

if not EXPORT_PATH.exists():
    raise FileNotFoundError(
        f"Export file not found: {EXPORT_PATH}. Update EXPORT_PATH to your local file."
    )

with EXPORT_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

messages = data.get("messages", [])
chat_name = data.get("name", "<unknown>")
chat_type = data.get("type", "<unknown>")

print(f"Chat: {chat_name} ({chat_type})")
print(f"Total raw messages: {len(messages):,}")


## 3) Normalize Messages

In [ ]:
records = []

for msg in messages:
    msg_date = msg.get("date")
    dt = datetime.fromisoformat(msg_date) if msg_date else None
    text = flatten_text(msg.get("text", ""))

    entities = msg.get("text_entities") or []
    entity_links = 0
    if isinstance(entities, list):
        entity_links = sum(
            1
            for e in entities
            if isinstance(e, dict) and e.get("type") in {"link", "text_link"}
        )

    url_links = len(URL_RE.findall(text))
    link_count = max(entity_links, url_links)

    sender = msg.get("from") or msg.get("actor") or "<system>"
    msg_type = msg.get("type", "unknown")

    words = [w.lower() for w in WORD_RE.findall(text)]

    records.append({
        "id": msg.get("id"),
        "datetime": dt,
        "month": dt.strftime("%Y-%m") if dt else None,
        "type": msg_type,
        "sender": sender,
        "text": text,
        "word_count": len(words),
        "link_count": link_count,
        "hashtags": [h.lower() for h in HASHTAG_RE.findall(text)],
        "urls": URL_RE.findall(text),
        "words": words,
    })

print(f"Normalized records: {len(records):,}")
records[0] if records else {}


## 4) Core Metrics

In [ ]:
type_counts = Counter(r["type"] for r in records)
sender_counts = Counter(r["sender"] for r in records)
message_records = [r for r in records if r["type"] == "message"]
service_records = [r for r in records if r["type"] == "service"]

dates = [r["datetime"] for r in records if r["datetime"] is not None]
start_date = min(dates) if dates else None
end_date = max(dates) if dates else None

avg_words = (sum(r["word_count"] for r in message_records) / len(message_records)) if message_records else 0.0
messages_with_links = sum(1 for r in message_records if r["link_count"] > 0)
link_share = (messages_with_links / len(message_records) * 100.0) if message_records else 0.0

summary = {
    "chat_name": chat_name,
    "chat_type": chat_type,
    "total_messages": len(records),
    "content_messages": len(message_records),
    "service_messages": len(service_records),
    "date_range": f"{start_date.date() if start_date else 'N/A'} -> {end_date.date() if end_date else 'N/A'}",
    "unique_senders": len(sender_counts),
    "avg_words_per_message": round(avg_words, 2),
    "messages_with_links_pct": round(link_share, 2),
}

summary


## 5) Activity by Month

In [ ]:
monthly_counts = Counter(r["month"] for r in message_records if r["month"] is not None)

for month, count in sorted(monthly_counts.items()):
    print(f"{month}: {count}")

peak_month, peak_count = monthly_counts.most_common(1)[0] if monthly_counts else ("N/A", 0)
print()
print(f"Peak month: {peak_month} ({peak_count} messages)")


## 6) Content Signals

In [ ]:
hashtag_counts = Counter()
domain_counts = Counter()
word_counts = Counter()

stop_words = {
    "the", "and", "for", "with", "this", "that", "from", "have", "your", "you", "are",
    "это", "как", "что", "для", "или", "все", "она", "они", "его", "под", "при", "без",
    "так", "еще", "уже", "если", "есть", "быть", "чтобы", "когда", "только", "после", "перед",
}

for r in message_records:
    hashtag_counts.update(r["hashtags"])

    for url in r["urls"]:
        domain = urlparse(url).netloc.lower().removeprefix("www.")
        if domain:
            domain_counts[domain] += 1

    filtered_words = [
        w for w in r["words"]
        if len(w) >= 4 and not w.isdigit() and w not in stop_words
    ]
    word_counts.update(filtered_words)

print("Top hashtags:")
for tag, count in hashtag_counts.most_common(10):
    print(f"  {tag}: {count}")

print("\nTop domains:")
for domain, count in domain_counts.most_common(10):
    print(f"  {domain}: {count}")

print("\nTop words (len >= 4):")
for word, count in word_counts.most_common(20):
    print(f"  {word}: {count}")


## 7) Optional Plot

In [ ]:
try:
    import matplotlib.pyplot as plt

    months = sorted(monthly_counts)
    values = [monthly_counts[m] for m in months]

    plt.figure(figsize=(10, 4))
    plt.plot(months, values, marker="o")
    plt.xticks(rotation=60)
    plt.title("Monthly Message Activity")
    plt.xlabel("Month")
    plt.ylabel("Messages")
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print("matplotlib is not installed. Skip this cell or install matplotlib to see charts.")


## 8) Concise Takeaways

In [ ]:
top_sender, top_sender_count = sender_counts.most_common(1)[0] if sender_counts else ("N/A", 0)
top_hashtag, top_hashtag_count = hashtag_counts.most_common(1)[0] if hashtag_counts else ("N/A", 0)
top_domain, top_domain_count = domain_counts.most_common(1)[0] if domain_counts else ("N/A", 0)

takeaways = [
    f"Dataset: {len(records):,} entries ({len(message_records):,} messages + {len(service_records):,} service events).",
    f"Time span: {start_date.date() if start_date else 'N/A'} to {end_date.date() if end_date else 'N/A'}.",
    f"Most active month: {peak_month} with {peak_count} messages.",
    f"Top sender: {top_sender} ({top_sender_count:,} entries).",
    f"Link usage: {link_share:.2f}% of content messages include links.",
    f"Top hashtag/domain: {top_hashtag} ({top_hashtag_count}) / {top_domain} ({top_domain_count}).",
]

print("Concise takeaways:")
for line in takeaways:
    print(f"- {line}")
